# Phase 3: Model Training

In this phase, we train Random Forest Regressors for Car, Bus, and Metro travel time predictions.

In [ ]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import pickle

# Data preparation
df = pd.read_csv("../backend/data/raw/Banglore_traffic_Dataset.csv")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['hour'] = df['Date'].dt.hour.fillna(8).astype(int)
df['day_of_week'] = df['Date'].dt.dayofweek.fillna(0).astype(int)
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
df['is_peak']     = df['hour'].apply(lambda h: 1 if h in range(8, 10) or h in range(17, 20) else 0)
df['congestion_norm'] = df['Congestion Level'] / 100.0

base = df['Travel Time Index'] * 20
df['car_time']   = (base * (1 + df['congestion_norm'] * 0.8)).round(1)
df['bus_time']   = (base * (1 + df['congestion_norm'] * 1.1)).round(1)
df['metro_time'] = (base * (1 + df['congestion_norm'] * 0.2)).round(1)

FEATURES = ['hour', 'day_of_week', 'is_peak', 'is_weekend', 'congestion_norm']
MODS = ['car', 'bus', 'metro']

## Training Random Forest Regressors

Training triple models (one for each mode) for maximum accuracy.

In [ ]:
for mode in MODS:
    target = f'{mode}_time'
    X = df[FEATURES]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    print(f"Trained RF for {mode.upper()}: Score = {rf.score(X_test, y_test):.4f}")